# Qwen3.5-4B layer + threshold calibration (Colab)

Этот notebook адаптирован под **`Qwen/Qwen3.5-4B`** и гоняет три шага в одной Colab-сессии:

1. **Layer search** через `scripts/run_qwen_anchor_geometry_probe.py`
2. **Threshold sweep** через `scripts/calibrate_qwen_anchor_thresholds.py`
3. **Geometry + generation calibration** через `scripts/run_qwen_geometry_generation_calibration.py`

Модель `Qwen3.5-4B` мультимодальная, поэтому notebook ставит **свежий `transformers` с `main`** и использует **text-only prompts**.


> Рекомендуемый runtime: **GPU (T4 / L4 / A100)**.
>
> Для `Qwen/Qwen3.5-4B` нужен актуальный `transformers` (модель-карта Hugging Face это отдельно отмечает).
>
> Все артефакты сохраняются в отдельные файлы с префиксом `qwen35_4b_`, чтобы не затирать старые результаты.


In [ ]:
!nvidia-smi || true
!python --version
!pip install -q --upgrade pip
!pip install -q torch torchvision pillow accelerate einops pytest numpy nbformat "transformers @ git+https://github.com/huggingface/transformers.git@main"


In [ ]:
%cd /content
import os
import subprocess
from pathlib import Path

if not os.path.exists('/content/ABPT'):
    !git clone https://github.com/kharkilirov1/Anchor-engine.git ABPT
else:
    %cd /content/ABPT
    !git pull --ff-only

%cd /content/ABPT
print(subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
import json
from pathlib import Path

MODEL_NAME = 'Qwen/Qwen3.5-4B'
DEVICE = 'cuda'
MAX_LENGTH = 160
GEOM_JSON = '/content/ABPT/archive/qwen35_4b_anchor_geometry_probe.json'
GEOM_MD = '/content/ABPT/docs/research/qwen35_4b_anchor_geometry_report.md'
THRESH_JSON = '/content/ABPT/archive/qwen35_4b_anchor_threshold_calibration.json'
THRESH_MD = '/content/ABPT/docs/research/qwen35_4b_anchor_threshold_calibration.md'
CAL_JSON = '/content/ABPT/archive/qwen35_4b_geometry_generation_calibration.json'
CAL_MD = '/content/ABPT/docs/research/qwen35_4b_geometry_generation_calibration.md'

for path in [GEOM_JSON, GEOM_MD, THRESH_JSON, THRESH_MD, CAL_JSON, CAL_MD]:
    Path(path).parent.mkdir(parents=True, exist_ok=True)

print('MODEL_NAME =', MODEL_NAME)
print('DEVICE =', DEVICE)


## 1) Layer search / geometry probe

Этот шаг прогоняет все anchor geometry cases по **всем hidden layers модели** и сохраняет полный профиль.


In [ ]:
!python scripts/run_qwen_anchor_geometry_probe.py --model-name "$MODEL_NAME" --device "$DEVICE" --max-length $MAX_LENGTH --json-path "$GEOM_JSON" --markdown-path "$GEOM_MD"


In [ ]:
geom = json.loads(Path(GEOM_JSON).read_text(encoding='utf-8'))
print('layers:', geom['layers'])
print('n_results:', len(geom['results']))
print('interpretation summary:')
print(json.dumps(geom['interpretation'], indent=2, ensure_ascii=False)[:4000])
print()
print('report head:')
print(Path(GEOM_MD).read_text(encoding='utf-8')[:5000])


## 2) Threshold sweep

Этот шаг кэширует hidden states и перебирает конфигурации anchor thresholds поверх `Qwen/Qwen3.5-4B`.


In [ ]:
!python scripts/calibrate_qwen_anchor_thresholds.py --model "$MODEL_NAME" --device "$DEVICE" --max_length 192 --output_json "$THRESH_JSON" --output_md "$THRESH_MD"


In [ ]:
thresholds = json.loads(Path(THRESH_JSON).read_text(encoding='utf-8'))
best = thresholds['evaluations'][0]
print('best threshold config:')
print(json.dumps(best['config'], indent=2, ensure_ascii=False))
print('score =', best['score'])
print('pressure_gap =', best['pressure_gap'])
print('viability_gap =', best['viability_gap'])
print()
print('threshold report head:')
print(Path(THRESH_MD).read_text(encoding='utf-8')[:5000])


## 3) Geometry + generation calibration

Здесь запускается generation calibration уже с **динамически выбранным tail-layer окном** для `Qwen3.5-4B`.


In [ ]:
!python scripts/run_qwen_geometry_generation_calibration.py --model "$MODEL_NAME" --device "$DEVICE" --max_length 160 --max_new_tokens 120 --output_json "$CAL_JSON" --output_md "$CAL_MD"


In [ ]:
cal = json.loads(Path(CAL_JSON).read_text(encoding='utf-8'))
print('metadata:')
print(json.dumps(cal['metadata'], indent=2, ensure_ascii=False))
print()
print('threshold candidates:')
print(json.dumps(cal['calibration']['threshold_candidates'], indent=2, ensure_ascii=False))
print()
for cluster, summary in cal['calibration']['by_cluster_clean_base'].items():
    print(cluster, json.dumps(summary, ensure_ascii=False))
print()
print('report head:')
print(Path(CAL_MD).read_text(encoding='utf-8')[:6000])


In [ ]:
# Optional: download artifacts from Colab runtime
# from google.colab import files
# for artifact in [GEOM_JSON, GEOM_MD, THRESH_JSON, THRESH_MD, CAL_JSON, CAL_MD]:
#     files.download(artifact)
